# Qriton HLM — Concept Surgery

Capture what a concept looks like inside a model, then inject it as a new attractor.
Blend concepts. Transplant between models. Export/import portable `.concept` files.

**Requires:** An HLM3 checkpoint with tokenizer.

In [ ]:
import torch
from qriton_hlm import BasinSurgeon

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH = 'model.pt'  # <- your HLM3 checkpoint

## 1. Load model

In [ ]:
surgeon = BasinSurgeon.from_checkpoint(MODEL_PATH, device=DEVICE)
# surgeon.load_model()  # enables text generation + capture

## 2. Capture — extract a concept's energy signature

Run multiple examples through the model. The Hopfield layer converges
each input to a basin. Averaging those basins gives the concept's centroid.

In [ ]:
# Capture several examples of 'polite' language
polite_texts = [
    'Thank you so much for your help',
    'I really appreciate your patience',
    'That is very kind of you',
    'Would you mind helping me with this',
    'I am grateful for your assistance',
]

for text in polite_texts:
    result = surgeon.capture(layer=5, text=text, concept_name='polite')
    print(f'  E={result["energy"]:.4f} basin={result["is_basin"]} "{text[:40]}"')

print(f'\nConcept "polite": {surgeon.list_concepts()["polite"]} samples')

In [ ]:
# Capture 'formal' language
formal_texts = [
    'Per our previous correspondence',
    'Please find attached the requested documents',
    'I am writing to inform you that',
    'In accordance with the regulations',
]

for text in formal_texts:
    surgeon.capture(layer=5, text=text, concept_name='formal')

print(surgeon.list_concepts())

## 3. Inject concept — create a new attractor

In [ ]:
result = surgeon.inject_concept(layer=5, concept_name='polite', strength=0.1)

print(f'Existed before: {result["existed_before"]} (cos={result["cos_before"]:.4f})')
print(f'Exists after:   {result["exists_after"]} (cos={result["cos_after"]:.4f})')
print(f'Samples used:   {result["num_samples"]}')

## 4. Blend concepts

In [ ]:
# 60% polite + 40% formal = 'professional'
blend = surgeon.blend('polite', 'formal', 'professional', ratio=0.6)
print(blend)

# Inject the blended concept
result = surgeon.inject_concept(layer=5, concept_name='professional', strength=0.1)
print(f'Professional basin exists: {result["exists_after"]}')

## 5. Apply to live model & test

In [ ]:
surgeon.apply(layer=5)

# Benchmark impact
bench = surgeon.benchmark()
print(f'Perplexity after surgery: {bench["perplexity"]:.2f}')

## 6. Export / Import concepts

In [ ]:
# Export
surgeon.export_concept('polite', 'polite.concept')
surgeon.export_concept('professional', 'professional.concept')

# Import in another session
# other_surgeon.import_concept('polite.concept')
# other_surgeon.inject_concept(layer=5, concept_name='polite')

## 7. Remove concept

In [ ]:
result = surgeon.remove_concept(layer=5, concept_name='polite', strength=0.2)
print(f'Basin still exists: {result["exists_after"]}')

# Restore everything
surgeon.restore_all()
print('All layers restored.')

---
**Next:** `03_landscape_visualization.ipynb` — interactive energy landscape plots